# 01 — Traitement des données

Ce notebook traite des points suivants :

1. Chargement des données brutes de foudre
2. Attribution des groupes d’orages et calcul de la cible
3. Nettoyage des anomalies et du bruit
4. Extraction des caractéristiques des impacts bruts
5. Construction de la grille temporelle minute par minute
6. Enregistrement de la grille traitée pour l’entraînement et pour les tests

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import pandas as pd
import numpy as np

from config.config import (
    ALERT_ZONE_KM,
    SAFETY_ZONES_KM,
    CONTEXT_SIZE,
    CONTEXT_WINDOW_MIN,
    HORIZON_MIN,
    AIRPORT_MAPPING
)

# add_time_to_end_target supprimé (leakage)
from src.preprocessing.storm_groups import add_storm_groups_by_target

from src.preprocessing.cleaning import (
    format_date, format_alert_id, format_last_lightning,
    format_icloud, remove_pise_2016, remove_noise_data, filter_useful_storms
)

# On importe uniquement les 5 fonctions de base utiles à la grille
from src.preprocessing.features import (
    sort_for_sequences,
    add_cartesian_coordinates,
    add_kinematic_features,
    add_intensity_features,
    add_cumulative_features
)

from src.temporal_grid.grid_builder import build_temporal_grid

## 1. Chargement des données brutes de foudre

In [2]:
df = pd.read_csv('../data/segment_alerts_all_airports_train.csv')
print(f"Loaded {len(df):,} rows — {df['airport'].nunique()} airports")
df.head(2)

Loaded 507,071 rows — 5 airports


,lightning_id,lightning_airport_id,date,lon,lat,amplitude,maxis,icloud,dist,azimuth,airport,airport_alert_id,is_last_lightning_cloud_ground
0,1,1,2016-01-02 14:53:36+00:00,9.0559,42.0826,-9.90,0.3,False,27.360653,57.852343,Ajaccio,NaN,NaN
1,2,2,2016-01-02 14:53:36+00:00,9.0236,42.0953,-3.33,0.2,True,26.383167,52.117828,Ajaccio,NaN,NaN


In [3]:
df_test = pd.read_csv('../data/segment_alerts_all_airports_test.csv')
print(f"Loaded {len(df_test):,} rows — {df_test['airport'].nunique()} airports")
df_test.head(2)

Loaded 188,175 rows — 5 airports


,lightning_id,lightning_airport_id,date,lon,lat,amplitude,maxis,icloud,dist,azimuth,airport,airport_alert_id,is_last_lightning_cloud_ground
0,72502,72502,2023-01-08 22:23:16+00:00,9.0451,42.0114,-123.04,0.257,False,22.262351,70.073897,Ajaccio,NaN,NaN
1,72503,72503,2023-01-08 22:25:20+00:00,9.0685,42.0267,-125.94,0.104,False,24.752225,68.784898,Ajaccio,NaN,NaN


## 2. Attribution des groupes d’orages et calcul de la cible

In [4]:
# train
format_date(df)
add_storm_groups_by_target(df, context_size=CONTEXT_SIZE, time_window_minutes=CONTEXT_WINDOW_MIN)

Storm groups assigned: 2627 groups found.


In [5]:
# test
format_date(df_test)
add_storm_groups_by_target(df_test, context_size=CONTEXT_SIZE, time_window_minutes=CONTEXT_WINDOW_MIN)

Storm groups assigned: 1081 groups found.


## 3. Nettoyage des anomalies et du bruit

In [6]:
format_alert_id(df)
format_last_lightning(df)
format_icloud(df)

remove_noise_data(df)
filter_useful_storms(df, alert_zone_km=ALERT_ZONE_KM)

Nettoyage : 69080 lignes de bruit supprimées. Restant : 437991
Filtrage : 2627 -> 2627 groupes | 437991 -> 437991 lignes


In [7]:
format_alert_id(df_test)
format_last_lightning(df_test)
format_icloud(df_test)

remove_noise_data(df_test)
filter_useful_storms(df_test, alert_zone_km=ALERT_ZONE_KM)

Nettoyage : 32568 lignes de bruit supprimées. Restant : 155607
Filtrage : 1081 -> 1081 groupes | 155607 -> 155607 lignes


## 4. Extraction des caractéristiques des impacts bruts

In [8]:
sort_for_sequences(df)
add_kinematic_features(df)
add_intensity_features(df)
add_cumulative_features(df)

In [9]:
sort_for_sequences(df_test)
add_kinematic_features(df_test)
add_intensity_features(df_test)
add_cumulative_features(df_test)

In [11]:
# Airport integer encoding (for LabelEncoder compatibility)
df['airport_id'] = df['airport'].map(AIRPORT_MAPPING)
print('Airport mapping:', AIRPORT_MAPPING)
# Cartesian coordinates
add_cartesian_coordinates(df)

Airport mapping: {'Bron': 0, 'Bastia': 1, 'Ajaccio': 2, 'Nantes': 3, 'Pise': 4, 'Biarritz': 5}


In [12]:
# Airport integer encoding (for LabelEncoder compatibility)
df_test['airport_id'] = df_test['airport'].map(AIRPORT_MAPPING)
print('Airport mapping:', AIRPORT_MAPPING)

# Cartesian coordinates
add_cartesian_coordinates(df_test)

Airport mapping: {'Bron': 0, 'Bastia': 1, 'Ajaccio': 2, 'Nantes': 3, 'Pise': 4, 'Biarritz': 5}


## 5. Construction de la grille temporelle minute par minute

In [14]:
df_grid = build_temporal_grid(
    df,
    horizon_min=HORIZON_MIN,
    safety_zones_km=SAFETY_ZONES_KM,
)
print(f"Train grid shape: {df_grid.shape}")
df_grid.head(2)

Construction de la grille temporelle (horizon=30m, zones=[3, 5, 7, 10, 15, 20]km)...
Assemblage de la grille finale...
Train grid shape: (209254, 44)


,date,airport_id,storm_group_id,minutes_since_last_strike,cum_n_strikes,activity_count_last_5m,dist_last_5m,delta_dist_last_5m,abs_amplitude_last_5m,activity_count_last_20m,...,target_30m_7km,target_30m_10km,target_30m_15km,target_30m_20km,count_cg_3km,count_cg_5km,count_cg_7km,count_cg_10km,count_cg_15km,count_cg_20km
0,2016-01-06 02:40:00+00:00,1,531,0.0,1.0,1.0,25.68437,0.000000,25.36,1.0,...,0,0,0,1,0,0,0,0,0,0
1,2016-01-06 02:41:00+00:00,1,531,0.0,2.0,2.0,25.68437,0.398996,25.36,2.0,...,0,0,0,1,0,0,0,0,0,0


In [15]:
df_grid_test = build_temporal_grid(
    df_test,
    horizon_min=HORIZON_MIN,
    safety_zones_km=SAFETY_ZONES_KM,
)
print(f"Test grid shape: {df_grid_test.shape}")
df_grid_test.head(2)

Construction de la grille temporelle (horizon=30m, zones=[3, 5, 7, 10, 15, 20]km)...
Assemblage de la grille finale...
Test grid shape: (87105, 44)


,date,airport_id,storm_group_id,minutes_since_last_strike,cum_n_strikes,activity_count_last_5m,dist_last_5m,delta_dist_last_5m,abs_amplitude_last_5m,activity_count_last_20m,...,target_30m_7km,target_30m_10km,target_30m_15km,target_30m_20km,count_cg_3km,count_cg_5km,count_cg_7km,count_cg_10km,count_cg_15km,count_cg_20km
0,2023-01-09 18:36:00+00:00,1,204,0.0,1.0,1.0,12.885839,0.0,62.09,1.0,...,0,0,1,1,0,0,0,0,1,1
1,2023-01-09 18:37:00+00:00,1,204,1.0,1.0,1.0,12.885839,0.0,62.09,1.0,...,0,0,1,1,0,0,0,0,0,0


## 6. Enregistrement de la grille traitée pour l’entraînement

In [16]:
OUTPUT_PATH = '../data/grid_train.parquet'
df_grid.to_parquet(OUTPUT_PATH, index=False)
print(f"Grid saved to {OUTPUT_PATH}")

Grid saved to ../data/grid_train.parquet


In [17]:
OUTPUT_PATH = '../data/grid_test.parquet'
df_test.to_parquet(OUTPUT_PATH, index=False)
print(f"Grid saved to {OUTPUT_PATH}")

Grid saved to ../data/grid_test.parquet
